## Note: The case_briefs and transcripts folder are pre-cleaned! This class will error out if you run it on data download directly from Oyez.

In [1]:
import json
import os
import re

In [4]:
def substring_in_list(main_string, list_of_substrings):
    for substring in list_of_substrings:
        if substring in main_string:
            return True
    return False

# Phrases that manage the flow of conversation but aren't substantive.
# Will be removed if a turn consists solely of one of these.
TRAFFIC_PHRASES = {
    "i'm sorry.", "go ahead.", "no, please.", "thank you.", "yes.",
    "okay.", "all right.", "please.", "you're welcome"
}

# Short acknowledgements that can be removed if they are between two turns from the same speaker.
SIMPLE_INTERJECTIONS = {
    "yeah.", "right.", "mm-hmm.", "sure.", "no.", "correct.", "yes.", "uh-huh.", 
    "oh."
}

This transcript cleaner will broadly do three things:
1. Remove excess info from the Oyez transcripts
    a. consolidate text blocks for each turn
    b. simpifies stored speaker information
    c. recombines petitioner/respondent "side" information for each advocate
2. Remove certain turns that appear to be interjections, false starts, or "traffic management" terms that don't contribute to the source value fo the transcript
3. Partion each transcript into sections where each section starts with an advocate opening statement and ends at the conclusion of questioning.

In [5]:
class TranscriptCleaner:
    def __init__(self, transcript_json, advocate_mapping, transcript_year):
        self.transcript_year = transcript_year
        self.processed_turns = self._get_all_processed_turns(transcript_json, advocate_mapping)
        self.indices_to_delete = set()    


    def get_cleaned_transcript_sections(self, clean_turn_overlaps: bool):
        ''' 
        Reprocesses a SCOTUS oral transcript into cleaned sections, where each section
        begins with an opening statement and ends with the conclusion of one side's argument.
        '''
        if clean_turn_overlaps:
            self._flag_traffic_management_turns()
            self._flag_interrupted_false_starts()
            self._flag_simple_interjections()

        cleaned_turns = self._build_cleaned_list()
        merged_turns = self._merge_consecutive_turns(cleaned_turns)
        return self._segment_turns_into_sections(merged_turns)
    

    def _get_all_processed_turns(self, transcript_json, advocate_mapping):
        all_processed_turns = []
        for section in transcript_json.get("transcript", {}).get("sections", []):
            for turn in section.get("turns", []):
                # 1. Consolidate 'text_block' into one string
                consolidated_text = " ".join([block.get('text', '') for block in turn.get('text_blocks', [])])

                # 2. Consolidate 'speaker' information, return empty dictionary if speaker info in original transcript is null
                speaker_info = self._create_speaker_info_dict(turn, advocate_mapping)

                # 3. Format new turn and append to the new turn list
                processed_turn = {
                'start': turn.get('start'),
                'stop': turn.get('stop'),
                'speaker': speaker_info,
                'text': consolidated_text.strip()
                }
                all_processed_turns.append(processed_turn)
        return all_processed_turns
    

    def _assign_speaker_as_advocate(self, speaker_info, speaker_id, advocate_mapping):
        speaker_info['role'] = 'advocate' # Default if role is null (is not petitioner or respondent already) or not found
        if advocate_mapping:
            speaker_info['side'] = advocate_mapping.get(speaker_id)
        else:
            speaker_info['side'] = "inferred"
    

    def _create_speaker_info_dict(self, turn, advocate_mapping):
        # 1. First handle the case where the speaker is null
        speaker = turn.get('speaker')
        if speaker is None:
            return {
                "name": "unspecified",
                "ID": "xxxxxxx",
                "role": "inferred",
                "side": "inferred"
            }

        # 2. Initialize speaker info
        speaker_info = {}
        speaker_info['name'] = speaker.get('name')
        speaker_info['ID'] = speaker.get("ID")
        
        # 3. Catches Justice Barrett's strange role formatting and assign her role to "scotus_justice"
        roles = speaker.get('roles', {})
        if roles and '2' in roles:
            speaker_info['role'] = roles['2']['type']
            return speaker_info

        # 4. Assigns John G. Roberts Jr. to advocate if he appears before the Court prior to his
        # 2005 swearing-in
        if roles and speaker_info['ID'] == 15086 and self.transcript_year < 2005:
            self._assign_speaker_as_advocate(speaker_info, speaker_info['ID'], advocate_mapping)
            return speaker_info

        # 5. Assigns Elena Kagan to advocate if she appears before the Court prior to her
        # 2010 swearing-in
        if roles and speaker_info['ID'] == 15094 and self.transcript_year < 2010:
            self._assign_speaker_as_advocate(speaker_info, speaker_info['ID'], advocate_mapping)
            return speaker_info

        # 6. Assigns Ketanji Brown Jackson to advocate if she appears before the Court prior to her
        # 2022 swearing-in
        if roles and speaker_info['ID'] == 33869 and self.transcript_year < 2022:
            self._assign_speaker_as_advocate(speaker_info, speaker_info['ID'], advocate_mapping)
            return speaker_info

        # 7. Assigns Samuel Alito to advocate if he appears before the Court prior to his
        # 2006 swearing-in
        if roles and speaker_info['ID'] == 15068 and self.transcript_year < 2006:
            self._assign_speaker_as_advocate(speaker_info, speaker_info['ID'], advocate_mapping)
            return speaker_info

        # 8. Assigns Brett Kavanaugh to advocate if he appears before the Court prior to his
        # 2018 swearing-in
        if roles and speaker_info['ID'] == 17766 and self.transcript_year < 2018:
            self._assign_speaker_as_advocate(speaker_info, speaker_info['ID'], advocate_mapping)
            return speaker_info

        # 10. Assign the rest of the justices and speakers to their proper sides
        if roles and len(roles) > 0 and roles[0].get('type'):
            # the speaker is a justice
            speaker_info['role'] = roles[0]['type']
        else:
            # the speaker is an advocate
            self._assign_speaker_as_advocate(speaker_info, speaker_info['ID'], advocate_mapping)
        return speaker_info
    

    def _get_next_turn_index(self, i):
        next_turn_index = -1
        for j in range(i + 1, len(self.processed_turns)):
            if j not in self.indices_to_delete:
                next_turn_index = j
                break
        return next_turn_index
    

    def _get_previous_turn_index(self, i):
        prev_turn_index = -1
        for j in range(i - 1, -1, -1):
            if j not in self.indices_to_delete:
                prev_turn_index = j
                break
        return prev_turn_index
    

    def _flag_traffic_management_turns(self):
        """Pass 1: Flag turns that are just conversational traffic management or false starts."""
        for i, turn in enumerate(self.processed_turns):
            text = turn["text"].lower()
            
            # Remove laughter-only turns
            if text == "(laughter.)" or text == '[inaudible]':
                self.indices_to_delete.add(i)

            # Remove phrases used to cede the floor or manage flow
            if text in TRAFFIC_PHRASES:
                self.indices_to_delete.add(i)
    

    def _flag_interrupted_false_starts(self):
        """Pass 2: Flag turns that are short, interrupted false starts."""
        for i, turn in enumerate(self.processed_turns):
            if i in self.indices_to_delete:
                continue

            text = turn["text"].lower()
            duration = turn['stop'] - turn['start']

            # A turn is a false start if it's short and ends with a dash.
            if duration < 2.0 and text.endswith('--'):
                # Find the next non-deleted turn to see who speaks next.
                next_speaker_index = self._get_next_turn_index(i)

                if next_speaker_index != -1:
                    # If the next speaker is different, this was a true interruption.
                    if self.processed_turns[i]['speaker']['name'] != self.processed_turns[next_speaker_index]['speaker']['name']:
                        self.indices_to_delete.add(i)


    def _flag_simple_interjections(self):
        """Pass 3: Flag simple interjections from a listener to a speaker who retains the floor."""
        for i, turn in enumerate(self.processed_turns):
            if i in self.indices_to_delete:
                continue

            text = turn["text"].lower()
            if text in SIMPLE_INTERJECTIONS:
                # Find the previous and next non-deleted turns
                prev_turn_index = self._get_previous_turn_index(i)
                next_turn_index = self._get_next_turn_index(i)

                # If the speaker before and after the interjection is the same, remove it.
                if (prev_turn_index != -1 and next_turn_index != -1 and
                        self.processed_turns[prev_turn_index]['speaker']['name'] == self.processed_turns[next_turn_index]['speaker']['name']):
                    self.indices_to_delete.add(i)


    def _merge_consecutive_turns(self, cleaned_turns):
        """Pass 4: Merge adjacent turns from the same speaker."""
        merged_turns = [cleaned_turns[0]]
        for i in range(1, len(cleaned_turns)):
            current_turn = cleaned_turns[i]
            last_merged_turn = merged_turns[-1]

            # name doesn't exist if speaker is null 
            current_speaker_name = current_turn["speaker"]["name"]
            last_speaker_name = last_merged_turn["speaker"]["name"]
            if current_speaker_name == last_speaker_name:
                # Merge: append text blocks and update stop time
                last_merged_turn['text'] += " " + current_turn['text']
                last_merged_turn['stop'] = current_turn['stop']
            else:
                # Different speaker, just append
                merged_turns.append(current_turn)
        return merged_turns
    

    def _build_cleaned_list(self):
        """Builds the initial list of turns, excluding the flagged ones."""
        return [turn for i, turn in enumerate(self.processed_turns) if i not in self.indices_to_delete]
    

    def _split_last_section_at_index(self, sections, last_section, rebuttal_turn_start_idx):
        current_section_turns = []
        for i, turn in enumerate(last_section["turns"]):
            if i == rebuttal_turn_start_idx:
                last_section["stop"] = turn["stop"]
                last_section["turns"] = current_section_turns
                sections.append(last_section)

                # re-initialize for the rebuttal turns
                last_section["start"] = turn["start"]
                current_section_turns = []
            current_section_turns.append(turn)
        
         # add rebuttal turn to sections: 
        # last_section["stop"] = turn["stop"]
        # last_section["turns"] = current_section_turns
        # sections.append(last_section)
        return sections
        

    def _reprocess_sections_to_remove_rebuttal(self, sections):
        current_advocate = None
        last_section = sections.pop()
        rebuttal_turn_start = -1
        # Split off rebuttal from the last section splitting at the first speaker change, if it exists, 
        # going from the end of the transcript up.
        for i, turn in enumerate(reversed(last_section["turns"])):
            speaker_name = turn["speaker"].get("name", "")
            speaker_role = turn["speaker"].get("role", "")
            if speaker_role == 'advocate' and current_advocate == None:
                current_advocate = speaker_name

            if speaker_role == 'advocate' and current_advocate != speaker_name:
                # found the rebuttal turn! Do some maths to get the 
                # rebuttal turn start index when going chronologically
                rebuttal_turn_start = len(last_section["turns"]) - i + 1
                break
        
        if rebuttal_turn_start == -1:
            # there is no rebuttal, reattach last section and return
            sections.append(last_section)
            return sections

        return self._split_last_section_at_index(sections, last_section, rebuttal_turn_start)
    

    def _segment_turns_into_sections(self, merged_turns):
        merged_turns.pop(0) # remove the first turn where Robert's intro's the case
        reorganized_sections = []
        current_section = None
        current_advocate = None
        opening_statment_starts = ["it please the court", "good morning", "it please this court"]

        for turn in merged_turns:  
            speaker_name = turn["speaker"].get("name", "")
            turn_text = turn.get("text", "")

            # create new section on advocate change; don't check for role b/c roberts was an advocate
            if speaker_name != current_advocate and substring_in_list(turn_text.lower(), opening_statment_starts):
                if current_section is not None:
                    current_section["stop"] = turn["stop"]
                    reorganized_sections.append(current_section)

                current_advocate = speaker_name
                current_section = {
                    "turns": [],
                    "start": turn["start"] # Set start time of the new section
                }
            
            if current_section:
                current_section["turns"].append(turn)
    
        # Add the last section if it exists
        if current_section is not None:
            reorganized_sections.append(current_section)
        
        return self._reprocess_sections_to_remove_rebuttal(reorganized_sections)


Here we have helper functions to get the advocate information from the cases brief. Also provides a method for running main. The TranscriptCleaner really only cleans a single transcript, so this section provides the scaffolding for it to do its work.

In [6]:

def assign_advocate_id_to_side(advocate_mapping, advocate, side):
    try:
        advocate_mapping[advocate['advocate']["ID"]] = side
    except:
        try:
            advocate_mapping[advocate["ID"]] = side
        except:
            # catches 3 cases where a speaker is not defined in Oyez
            return None

def get_advocate_mapping(case_brief_data):
    petitioner_words = ["petitioner", "appellant", "plaintiff"]
    respondent_words = ["respondent", "appellee", "defendant"]

    if case_brief_data["advocates"] is None:
        return None
    
    advocate_mapping = {}
    for advocate in case_brief_data["advocates"]:
        advocate_description = advocate['advocate_description'].lower()
        if substring_in_list(advocate_description, petitioner_words):
            assign_advocate_id_to_side(advocate_mapping, advocate, "petitioner")
        elif substring_in_list(advocate_description, respondent_words):
            assign_advocate_id_to_side(advocate_mapping, advocate, "respondent")

    return advocate_mapping

def get_brief_and_transcripts(docket_id):
    with open(f"./case_briefs/{docket_id}.json", 'r') as f:
        brief_data = json.load(f)

    transcript_links = brief_data["oral_argument_audio"]
    transcripts = []
    for num_transcript in range(1, len(transcript_links) + 1):
        with open("./transcripts/{}-t{:0>2d}.json".format(docket_id, num_transcript)) as t_file:
            transcripts.append(json.load(t_file))
        
    return brief_data, transcripts

def get_case_dockets():
    dockets = []
    for file in os.listdir("./case_briefs/"):
        match = re.search(r"^\d{4}\.[a-zA-Z0-9_-]+(?=\.json)", file)
        if match:
            dockets.append(match.group())
    return dockets

Main method! Runs through all the Oyez case briefs and transcripts to generate our cleaned versions.

In [8]:
for docket_id in get_case_dockets():
    case, transcripts = get_brief_and_transcripts(docket_id)
    advocate_mapping = get_advocate_mapping(case)
    

    sections = []
    for transcript in transcripts:
        cleaner = TranscriptCleaner(transcript, advocate_mapping, int(docket_id[:4]))
        
        #!important: Set clean_turn_overlaps to False if you don't want to remove any text information!
        new_sections = cleaner.get_cleaned_transcript_sections(clean_turn_overlaps=False)

        sections.extend(new_sections)

    new_transcript = {
        "title": case["name"],
        "sections": sections
    }

    #!important: Replace directory with the directory you want to write to!
    with open(f"./cleaned_transcripts/restructured_text_unchanged/{docket_id}-transcript.json", "w") as file:
        json.dump(new_transcript, file, indent=4)